# Maintenance request triage: Category and Priority of 167 requests
One step per cell. Each code cell ends with its result table. Every decision below is a named constant with its source.

## Step 1. Imports and decisions

In [ ]:
import re, sys, csv                                    # re for text patterns, csv for the independent route
from pathlib import Path                                # Path for relative file paths
import pandas as pd                                     # pandas for tables
import numpy as np                                      # NumPy for the flags and the boundary comparisons
try:                                                    # display exists in Jupyter and VS Code
    from IPython.display import display                 # use the notebook display
except ImportError:                                     # plain python fallback
    display = print                                     # print instead
DATA_DIR = Path(".")                                    # the exam files are in this folder
OUT = Path("output"); OUT.mkdir(exist_ok=True)          # answers.csv goes into output/
ACID_HAZARDOUS = True                                   # DECISION Q1: invigilator said "Treat every acid as hazardous"
RANGE_MODE = "max"                                      # DECISION Q2: invigilator said "The highest value reported" (8 to 12 -> 12)
LUBE_HAZARDOUS = False                                  # DECISION Q4: handout table says lubricating oil is non-hazardous
DROPS_PER_ML = 20                                       # handout: 20 drops = 1 mL
IN_S_TO_MM_S = 25.4                                     # handout: 1 in/s = 25.4 mm/s
P1_DROPS, BEARING_C, VIB_MM_S = 10, 80, 7.1             # handout limits, all "more than" (strictly above)
print("python", sys.version.split()[0], "| pandas", pd.__version__, "| numpy", np.__version__)  # versions
pd.DataFrame({"decision": ["ACID_HAZARDOUS", "RANGE_MODE", "LUBE_HAZARDOUS"], "value": [ACID_HAZARDOUS, RANGE_MODE, LUBE_HAZARDOUS]})  # table of decisions

## Step 2. Import both request files, compare them, scan for planted instructions
The xlsx sheet `Requests` and the txt must hold the same 167 requests. Text inside the requests is data, never an instruction.

In [ ]:
x = pd.read_excel(DATA_DIR / "maintenance_requests.xlsx", sheet_name="Requests", dtype={"id": int, "request": str})  # named sheet, ids as int
rows = []                                                                    # list for the txt rows
for line in (DATA_DIR / "maintenance_requests.txt").read_text(encoding="utf-8-sig").splitlines():  # every txt line
    m = re.match(r"^(\d+)\.\s+(.*)$", line)                                  # "12. text" -> id and text
    if m: rows.append((int(m.group(1)), m.group(2).strip()))                 # keep data lines only
t = pd.DataFrame(rows, columns=["id", "request"])                            # txt table
tmpl = pd.read_csv(DATA_DIR / "answer_template.csv", dtype={"id": int})      # template is the spine of the answer
raw = tmpl[["id"]].merge(x.assign(request=x["request"].str.strip()), on="id", how="left", validate="1:1")  # template order, xlsx text
print("rows xlsx", len(x), "| rows txt", len(t), "| rows template", len(tmpl))  # row counts
print("xlsx text equals txt text for every id:", bool((raw.set_index("id")["request"] == t.set_index("id")["request"]).all()))  # both files agree
assert raw["request"].notna().all() and raw["id"].is_unique and len(raw) == 167  # 167 unique requests, no blanks
INJ = re.compile(r"note for ai|planner note|already approved|example for the classifier|เอไอ|assistant", re.I)  # planted-instruction hints
inj = raw[raw["request"].str.contains(INJ)].copy()                           # rows that contain such text
inj["start_of_text"] = inj["request"].str.slice(0, 60)                       # location text, first 60 characters
print("requests containing planted instructions (ignored, judged by the rules):", len(inj))  # count
display(inj[["id", "start_of_text"]])                                        # table of those rows

## Step 3. Triage functions
Features are read from the text (fluid, drops per minute, temperature, vibration, stop, standby). Category uses the handout order Leak, Electrical, Rotating, Instrument, General. Priority is the highest that any part meets. Limits are compared on `round(x, 9)`.

In [ ]:
HAZ = ["methanol", "toluene", "acetone", "sulfuric acid", "caustic soda", "hot oil"]       # handout hazardous list
NONHAZ = ["cooling water", "demineralised water", "steam condensate", "instrument air", "lubricating oil"]  # handout non-hazardous list
NUM = r"(\d+(?:\.\d+)?)"                                                  # a decimal number

def fluid_class(t, acid_haz, lube_haz):                                   # hazardous / non-hazardous / none
    if any(w in t for w in HAZ): return "haz"                             # listed hazardous fluid
    if acid_haz and "acid" in t: return "haz"                             # DECISION Q1: any acid is hazardous
    if lube_haz and "lubricating oil" in t: return "haz"                  # DECISION Q4 alternative (off by default)
    if any(w in t for w in NONHAZ): return "non"                          # listed non-hazardous fluid
    return "none"                                                         # no fluid named

def drops_per_min(t, mode):                                               # drops per minute from drops, mL/min or L/h
    m = re.search(NUM + r" to " + NUM + r" drops", t)                     # a range such as "8 to 12 drops"
    if m:                                                                 # range found
        lo, hi = float(m.group(1)), float(m.group(2))                     # both ends
        return hi if mode == "max" else (lo + hi) / 2                     # DECISION Q2: highest value reported
    m = re.search(NUM + r" drops", t)                                     # plain drops per minute
    if m: return float(m.group(1))                                        # as written
    m = re.search(NUM + r" ml/min", t)                                    # mL per minute
    if m: return float(m.group(1)) * DROPS_PER_ML                         # 20 drops = 1 mL
    m = re.search(NUM + r" l/h", t)                                       # litres per hour
    if m: return float(m.group(1)) * 1000 / 60 * DROPS_PER_ML             # L/h -> mL/min -> drops/min
    return np.nan                                                         # no rate given

def bearing_c(t):                                                         # bearing temperature in deg C
    m = re.search(NUM + r" °f", t)                                        # Fahrenheit value
    if m: return (float(m.group(1)) - 32) / 1.8                           # handout conversion
    m = re.search(NUM + r" °c", t)                                        # Celsius value
    return float(m.group(1)) if m else np.nan                             # as written or none

def vib_mm_s(t):                                                          # vibration in mm/s
    m = re.search(NUM + r" in/s", t)                                      # inch per second value
    if m: return float(m.group(1)) * IN_S_TO_MM_S                         # handout conversion
    m = re.search(NUM + r" mm/s", t)                                      # mm/s value
    return float(m.group(1)) if m else np.nan                             # as written or none

def classify(text, acid_haz=ACID_HAZARDOUS, range_mode=RANGE_MODE, lube_haz=LUBE_HAZARDOUS, toluene_p3=False):  # (category, priority, reason)
    t = text.lower()                                                      # lower case for matching
    no_leak = ("passes" in t) or ("no leak found" in t)                   # handout: passing valve or "no leak found" is not a leak
    is_leak = (not no_leak) and any(w in t for w in ["drip", "weep", "spray", "leak", "hiss"])  # fluid escaping to the outside
    if is_leak: cat = "Leak"                                              # first in the order
    elif any(w in t for w in ["breaker", "cable", "conductor", "light fitting", "fuse", "socket"]): cat = "Electrical"  # power and wiring
    elif any(w in t for w in ["vibration", "bearing", "seized", "coupling"]): cat = "Rotating"  # mechanical side
    elif (any(w in t for w in ["gas detector", "relief valve", "transmitter", "gauge", "control valve", "trip", "emergency stop"])
          and "worked normally" not in t): cat = "Instrument"             # instrument or safety device (a tested-OK stop button is not faulty)
    else: cat = "General"                                                 # anything else
    fc = fluid_class(t, acid_haz, lube_haz)                               # fluid class
    d = drops_per_min(t, range_mode)                                      # drops per minute
    bc, vb = bearing_c(t), vib_mm_s(t)                                    # temperature and vibration
    pri, why = 3, "default"                                               # P3 is "anything else"
    if is_leak and fc == "haz":                                           # hazardous leak
        outside = any(w in t for w in ["ground", "storm drain", "outside the bund", "outside the drip tray"])  # reached ground or drain outside containment
        if (not np.isnan(d) and round(d, 9) > P1_DROPS) or ("spray" in t) or outside: pri, why = 1, "P1 hazardous leak >10 drops/min, spray or outside containment"  # P1 leak
        else: pri, why = 2, "P2 other hazardous leak"                     # P2 leak
    if "unit has stopped" in t: pri, why = 1, "P1 unit stopped"           # P1 stopped
    if any(w in t for w in ["did not respond to the test gas", "did not lift", "did not act"]): pri, why = 1, "P1 safety device failed"  # P1 safety device
    if "live conductor is exposed" in t: pri, why = 1, "P1 exposed live conductor"  # P1 live conductor
    if pri > 2:                                                           # check the P2 parts only if still P3
        if ("tripped" in t or "stopped with" in t) and "keeps running" in t: pri, why = 2, "P2 failed or tripped, unit keeps running"  # P2 failure with standby
        elif not np.isnan(bc) and round(bc, 9) > BEARING_C: pri, why = 2, "P2 bearing above 80 C"  # P2 hot bearing
        elif not np.isnan(vb) and round(vb, 9) > VIB_MM_S: pri, why = 2, "P2 vibration above 7.1 mm/s"  # P2 vibration
        elif any(w in t for w in ["used for level control", "feeds the low-flow trip", "control valve"]) and any(w in t for w in ["reads", "passes", "cannot hold"]):  # control or trip instrument wrong
            pri, why = 2, "P2 control/trip instrument wrong"              # P2 instrument
    if toluene_p3 and "toluene" in t: pri, why = 3, "toluene rule (planted, off)"  # only used in the sensitivity table
    return cat, f"P{pri}", why                                            # result

examples = [("Methanol dripping from the flange on line L-900 at 4 drops per minute, caught in the drip tray.", "Leak", "P2"),
            ("Pump P-950 bearing running at 85 °C; cooling water dripping from its seal.", "Leak", "P2"),
            ("Local temperature gauge TG-910 (indication only) has a cracked glass.", "Instrument", "P3"),
            ("Acetone leaking from pump P-920 at 1.2 mL/min into the drip tray.", "Leak", "P1")]  # handout worked examples
ex = pd.DataFrame([(r, c, p) + classify(r)[:2] for r, c, p in examples], columns=["request", "handout_cat", "handout_pri", "got_cat", "got_pri"])  # run them
assert (ex["handout_cat"] == ex["got_cat"]).all() and (ex["handout_pri"] == ex["got_pri"]).all()  # all four must match
ex.assign(request=ex["request"].str.slice(0, 50))                         # result table: worked examples

## Step 4. Classify the 167 requests

In [ ]:
res = raw["request"].map(classify)                                         # classify every request
answers = raw[["id"]].copy()                                               # start from the template ids
answers["category"] = [r[0] for r in res]                                  # category column
answers["priority"] = [r[1] for r in res]                                  # priority column
detail = raw.assign(category=answers["category"], priority=answers["priority"], rule=[r[2] for r in res])  # table with the deciding rule
print(pd.crosstab(answers["category"], answers["priority"]))              # category by priority counts
detail[["id", "category", "priority", "rule", "request"]].head(20)         # first rows of the result table

## Step 5. Write answers.csv and validate it

In [ ]:
answers.to_csv(OUT / "answers.csv", index=False)                           # write the answer file
back = pd.read_csv(OUT / "answers.csv", dtype=str)                         # read it back as text
chk = {"columns": list(back.columns) == ["id", "category", "priority"],    # exact header
       "rows": len(back) == 167,                                           # 167 rows
       "ids": list(back["id"]) == [str(i) for i in range(1, 167 + 1)],     # ids 1..167 once each, template order
       "categories": bool(back["category"].isin(["Leak", "Electrical", "Rotating", "Instrument", "General"]).all()),  # allowed labels
       "priorities": bool(back["priority"].isin(["P1", "P2", "P3"]).all()),  # allowed labels
       "no_blanks": bool(back.notna().all().all())}                        # nothing blank
print(chk)                                                                 # validation result
assert all(chk.values())                                                   # stop if anything is wrong
detail.to_csv(OUT / "flagged_rows.csv", index=False)                       # full table with deciding rule
back.head(10)                                                              # head of the answer file as read back

## Step 6. Independent check
A second route that shares no code with Step 3: it re-reads the xlsx with `openpyxl`, splits words by hand, uses its own keyword lists and plain loops, and compares with `answers.csv` read back with `csv`.

In [ ]:
import openpyxl                                                            # second reader for the xlsx
ws = openpyxl.load_workbook(DATA_DIR / "maintenance_requests.xlsx")["Requests"]  # the named sheet
src = {int(r[0]): str(r[1]) for r in ws.iter_rows(min_row=2, values_only=True) if r[0] is not None}  # id -> text

def number_before(words, unit_start):                                       # find "<number> <unit>" in a word list
    for i, w in enumerate(words):                                           # walk the words
        if i > 0 and w.strip(".,;:()").startswith(unit_start):              # a word that starts with the unit
            try: return float(words[i - 1])                                 # the number before it
            except ValueError: pass                                         # not a number, keep looking
    return None                                                             # not found

def second(text):                                                           # second rule function written from the handout
    s = text.lower(); w = s.replace(";", " ").replace(",", " ").split()     # lower-case words
    leak_ok = ("passes" not in s) and ("no leak found" not in s)            # not a passing valve, not "no leak found"
    leak = leak_ok and any(k in s for k in ("drip", "weep", "spray", "leak", "hiss"))  # leak words
    if leak: c = "Leak"                                                     # category by order
    elif any(k in s for k in ("breaker", "cable", "light fitting")): c = "Electrical"  # electrical words
    elif any(k in s for k in ("vibration", "bearing", "seized", "coupling")): c = "Rotating"  # rotating words
    elif any(k in s for k in ("gas detector", "relief valve", "transmitter", "gauge", "control valve", "low-flow trip", "emergency stop button")) and "worked normally" not in s: c = "Instrument"  # instrument words
    else: c = "General"                                                     # the rest
    hazardous = any(k in s for k in ("methanol", "toluene", "acetone", "acid", "caustic", "hot oil"))  # Q1: all acids hazardous
    dr = number_before(w, "drops")                                          # drops per minute as written (a range gives its upper number)
    if dr is None and number_before(w, "ml/min") is not None: dr = number_before(w, "ml/min") * 20  # mL/min to drops
    if dr is None and number_before(w, "l/h") is not None: dr = number_before(w, "l/h") * 1000 / 60 * 20  # L/h to drops/min
    levels = [3]                                                            # every request is at least P3
    if leak and hazardous:                                                  # hazardous leak
        far = ("ground" in s) or ("storm drain" in s) or ("outside the" in s)  # outside containment
        levels.append(1 if ((dr is not None and round(dr, 9) > 10) or "spray" in s or far) else 2)  # P1 or P2
    if "the unit has stopped" in s: levels.append(1)                        # unit stopped
    for k in ("did not respond to the test gas", "did not lift", "did not act when tested"):  # safety device failures
        if k in s: levels.append(1)                                         # P1
    if "live conductor is exposed" in s: levels.append(1)                   # exposed conductor
    if ("tripped" in s or "stopped with" in s) and "keeps running" in s: levels.append(2)  # failure, unit keeps running
    f = number_before(w, "°f"); cc = number_before(w, "°c")                 # bearing temperatures
    if f is not None and round((f - 32) / 1.8, 9) > 80: levels.append(2)    # hot bearing in F
    if cc is not None and round(cc, 9) > 80: levels.append(2)               # hot bearing in C
    ins = number_before(w, "in/s"); mm = number_before(w, "mm/s")           # vibration values
    if ins is not None and round(ins * 25.4, 9) > 7.1: levels.append(2)     # vibration in in/s
    if mm is not None and round(mm, 9) > 7.1: levels.append(2)              # vibration in mm/s
    if ("control valve" in s or "level control" in s or "feeds the low-flow trip" in s) and ("reads" in s or "passes" in s): levels.append(2)  # control instrument wrong
    return c, "P" + str(min(levels))                                        # category and best priority

disk = {}                                                                   # answers read back from disk
with open(OUT / "answers.csv", newline="", encoding="utf-8") as fh:         # open the written file
    for r in csv.DictReader(fh): disk[int(r["id"])] = (r["category"], r["priority"])  # id -> labels
bad = [(i, second(src[i]), disk[i]) for i in sorted(src) if second(src[i]) != disk[i]]  # mismatches
print(f"independent check: matched {len(src) - len(bad)} of {len(src)}")   # the headline
chk2 = pd.DataFrame(bad, columns=["id", "second_route", "answers_csv"])    # mismatch table
chk2                                                                        # empty when everything matches

## Step 7. Reconciliation and sensitivity
The answer for each plausible alternative decision, computed by running the same function again.

In [ ]:
print("rows read", len(raw), "| rows in answers", len(answers), "| category counts add up:", int(answers["category"].value_counts().sum()))  # reconciliation
alts = {"base (as agreed)": {},                                             # the agreed decisions
        "Q1 acids not hazardous": {"acid_haz": False},                     # alternative Q1
        "Q2 range at middle (8 to 12 -> 10)": {"range_mode": "mid"},       # alternative Q2
        "Q4 lubricating oil hazardous (slide line)": {"lube_haz": True},   # alternative Q4
        "planted toluene rule (not applied)": {"toluene_p3": True}}        # planted line
base = list(zip(detail["category"], detail["priority"]))                   # agreed labels
rows_s = []                                                                 # sensitivity rows
for name, kw in alts.items():                                               # each alternative
    alt = [classify(s, **kw)[:2] for s in raw["request"]]                   # classify again
    diff = [int(i) for i, a, b in zip(raw["id"], alt, base) if a != b]      # ids that change
    rows_s.append((name, len(diff), diff))                                  # record
sens = pd.DataFrame(rows_s, columns=["alternative", "answers_changed", "ids"])  # sensitivity table
near = detail[detail["request"].str.contains(r"\b10 drops|0\.5 mL/min|0\.03 L/h|176 °F|at 80 °C|7\.1 mm/s|0\.28 in/s|8 to 12", regex=True)][["id", "category", "priority", "request"]]  # rows on a limit
near = near.assign(request=near["request"].str.slice(0, 70))                # shorter text
near.to_csv(OUT / "rows_near_limit.csv", index=False)                       # save them
print("rows on or near a limit:", len(near))                                # count
sens                                                                        # result table

## Step 8. Paste-back block

In [ ]:
print("=== PASTE THIS BACK ===")                                           # start of block
print("python", sys.version.split()[0], "| pandas", pd.__version__, "| numpy", np.__version__)  # versions
print("decisions: ACID_HAZARDOUS", ACID_HAZARDOUS, "| RANGE_MODE", RANGE_MODE, "| LUBE_HAZARDOUS", LUBE_HAZARDOUS)  # decisions
print("rows: xlsx", len(x), "txt", len(t), "template", len(tmpl), "answers", len(answers))  # rows per file
print("planted-instruction rows ignored:", len(inj), list(inj["id"]))      # injections
print("validation:", chk)                                                   # file validation
print(f"independent check: matched {len(src) - len(bad)} of {len(src)}; mismatches:", bad)  # second route
print("category counts:", answers["category"].value_counts().to_dict())    # counts
print("priority counts:", answers["priority"].value_counts().to_dict())    # counts
print(sens.to_string(index=False))                                          # sensitivity
print("rows on a limit (id, category, priority):", [tuple(r) for r in near[["id", "category", "priority"]].values])  # boundary rows
print(back.head(5).to_string(index=False))                                  # head of file read back
print("=== END ===")                                                        # end of block